# CELL-FM — PLS generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BoHuangLab/CELL-FM/blob/master/notebooks/pls_generation.ipynb)

The other two notebooks run **CELL-FM** forwards, from sequence to image. This one runs it
**backwards**, and asks it to design a **protein localization signal**.

Show the model a cell whose protein sits where you want it — in the nucleus, or out in the
cytosol — then hand it a scaffold peptide with a blank tail and ask what residues would put
a protein *there*. Fill the blanks a few hundred times and the residues it reaches for are
its answer.

| Stage | What runs | Output |
| --- | --- | --- |
| **1 · Generate** | the model fills a masked tail against each anchor cell, one forward pass per residue | 320 peptides per signal type |
| **2 · Collect** | draws gathered across sixteen tail lengths | a table of candidate signals |
| **3 · Analyse** | residue frequencies against the human proteome | the enrichment figure |

Both signal types are generated, because the figure is a **comparison** — one anchor asking
for import, the other for export, everything else held fixed.

---

**You need a GPU runtime.** *Runtime → Change runtime type → T4 GPU*. Setup downloads about
5 GB once — 2.5 GB of CELL-FM weights and 2.3 GB for the ESM-C protein encoder.

**This is a long run.** The defaults are the reference settings — 16 tail lengths × 20 draws
× 2 signal types — which is 11,200 forward passes: **about 10 minutes on an A40**, measured,
and roughly 40 on a Colab T4 if the usual factor of four holds. That is the cost of
reproducing the published figure rather than sketching it; the sampling cell prints an
estimate first, and shortening `LENGTHS` or `DRAWS_PER_LENGTH` scales it down close to
linearly.

| | |
| --- | --- |
| Weights | [huggingface.co/BoHuangLab/CELL-FM](https://huggingface.co/BoHuangLab/CELL-FM) |
| Code | [github.com/BoHuangLab/CELL-FM](https://github.com/BoHuangLab/CELL-FM) |

## 1 · Setup

Check the runtime, install what Colab does not ship, fetch the code, weights and the two
anchor cells, then build the model. Run them once, in order.


In [ ]:
import subprocess
import sys

print("python  ", sys.version.split()[0])
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("gpu     ", gpu if gpu else
      "NONE — Runtime > Change runtime type > T4 GPU, then rerun this cell")


In [ ]:
# Colab already ships torch, numpy, pandas and matplotlib; this adds the rest of the stack.
# Two choices below look odd and are deliberate:
#
#   --no-deps on esm   its metadata requires torchtext, which has no wheel past Python
#                      3.11 and would drag torch backwards. Nothing on the ESM-C code
#                      path imports it — the second line is what the import closure
#                      actually needs, esm's other bounds included.
#   "transformers<4.48.2"  esm's own bound. It is behavioural, not cosmetic: 4.47 replaced
#                      the special-token properties on PreTrainedTokenizer with a
#                      _special_tokens_map served through __getattr__, and a tokenizer that
#                      misses the change returns mask_token = None, which kills generation
#                      inside esm/utils/encoding.py with "replace() argument 2 must be str,
#                      not None". esm 3.2 adapted to it; the bound is where that adaptation
#                      stops being tested.
#   no flash-attn      without it ESM-C falls back to a pure-torch rotary embedding,
#                      verified to give identical results, and skips a CUDA build.
#
# esm 3.1.4 used to be pinned here, and it forced a much worse install: it requires
# biotite==0.41.2, biotite 0.41 requires numpy<2, and biotite sits on the ESM-C import
# path — so the whole stack came down to NumPy 1.x. Under Colab's Python 3.13 neither
# numpy 1.26 nor biotite 0.41 has a wheel, so pip built both from source, and the numpy
# downgrade collided with every preinstalled Colab package that wants NumPy 2. esm 3.2
# moved to biotite>=1.0, which is NumPy 2 clean, and its ESM-C modules are byte-identical
# to 3.1.4's, so the checkpoint's tensor names are unchanged.
import sys

%pip install -q --no-deps --no-warn-conflicts "esm==3.2.1.post1"
%pip install -q --no-warn-conflicts "transformers<4.48.2" "diffusers==0.31.0" torchdiffeq loguru accelerate "biotite>=1.0" biopython msgpack-numpy cloudpathlib tenacity brotli zstd attrs einops tifffile

import importlib.metadata as md

# --no-warn-conflicts silences pip's post-install report, which here says only two things
# and both are expected:
#
#   "esm requires torchtext, which is not installed"   left unenforced on purpose by
#       --no-deps: nothing on the ESM-C code path imports it, and it has no wheel for this
#       Python. esm's transformers bound, by contrast, IS enforced above — that one is real.
#   "gradio requires huggingface-hub>=1.16"   transformers < 4.48.2 wants hub < 1.0 and
#       Colab preinstalls a gradio that wants a newer one. Nothing here imports gradio.
#
# The flag is safe here because it is not what verifies the install: the check below reads
# the installed versions, and the model cell asserts the tokenizer behaviour the pin exists
# to protect. Both are stronger than pip's declarative check, which only compares metadata.
print("\nversions")
for pkg in ("torch", "numpy", "pandas", "transformers", "diffusers", "esm", "biotite"):
    try:
        print(f"  {pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  {pkg:14s} MISSING")


def _version(pkg):
    """(major, minor, patch) for comparison; 4.48 sorts below 4.48.2, as it should."""
    return tuple(int(x) for x in md.version(pkg).split(".")[:3] if x.isdigit())


ready = _version("transformers") < (4, 48, 2)
print(f"\ntransformers {md.version('transformers')} < 4.48.2:",
      "yes — the imports below will work" if ready else
      "NO — the next cell will fail; rerun this one")

# A pin landing on disk is not the same as this kernel using it. If an earlier attempt got
# as far as the next cell then transformers is imported, and pip downgrades it underneath a
# kernel that goes on holding the old module in memory. So compare what is imported against
# what is installed and restart if they disagree. Expected, not a crash: Colab reconnects on
# its own and you carry on from the next cell, without rerunning this one.
stale = []
for pkg in ("transformers", "tokenizers", "huggingface_hub", "biotite"):
    loaded = getattr(sys.modules.get(pkg), "__version__", None)
    try:
        installed = md.version(pkg)
    except md.PackageNotFoundError:
        continue
    if loaded is not None and loaded != installed:
        stale.append(f"{pkg} {loaded} -> {installed}")

if stale:
    import os, time
    print("\nRestarting the kernel so these take effect — expected, not a crash:")
    for line in stale:
        print("   ", line)
    print("Colab reconnects by itself; continue from the next cell.")
    time.sleep(1)
    os.kill(os.getpid(), 9)


In [ ]:
import os
import sys
import time

# transformers probes for a TensorFlow backend at import time and imports it if present.
# Colab ships TensorFlow, nothing here uses it, and loading it costs seconds for nothing.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from huggingface_hub import hf_hub_download, snapshot_download

# esm 3.2 needs transformers < 4.48.2. Past that the special-token plumbing it adapted to
# moves again, tokenizer.mask_token comes back None, and generation dies inside esm on
# "replace() argument 2 must be str, not None" rather than at import.
if tuple(int(x) for x in transformers.__version__.split(".")[:3]) >= (4, 48, 2):
    raise RuntimeError(
        f"This kernel has transformers {transformers.__version__} loaded; esm 3.2 needs "
        "< 4.48.2. If the install cell above already pins it, the kernel is holding the "
        "old module: Runtime > Restart session, then run the cells again from the top."
    )

# The model code comes from the public Space, the same snapshot the other two notebooks
# use — CELLFMModel.image_to_sequence is already in that vendored subset, so this
# application needs no code of its own, only its own weights.
CODE = snapshot_download(repo_id="BoHuangLab/CELL-FM", repo_type="space")
sys.path.insert(0, CODE)

WEIGHTS_REPO = "BoHuangLab/CELL-FM"
hpa = lambda name: hf_hub_download(repo_id=WEIGHTS_REPO, filename=f"hpa/{name}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("code   ", CODE)
print("weights", WEIGHTS_REPO, "hpa/")
print("device ", DEVICE)

# One palette for every figure below.
INK, MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"
NUCLEAR, CYTO, MARK = "#2a78d6", "#eb6834", "#e53935"
plt.rcParams.update({
    "figure.dpi": 120, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
})

In [ ]:
from cell_fm.criterions.cell_fm.unidiffuser import UniDiffCriterions
from cell_fm.models.cell_fm.cell_fm_config import CELLFMConfig
from cell_fm.models.cell_fm.cell_fm_model import CELLFMModel
from esm.tokenization.sequence_tokenizer import EsmSequenceTokenizer
from esm.utils import encoding, decoding

# Hyperparameters transcribed from scripts/cell_fm/evaluate_img2seq_hpa.sh.
#
# This is the image-to-sequence model, and it is NOT the one the other HPA notebook loads.
# It runs at 512 px with sample_size=128, and it has its own VAE: pairing cellfm_img2seq
# with the 256 px vae.bin is a latent-size mismatch, not a resize. The two differ in five
# more places too — encoder_patch_size 8, img_generator_patch_size 4, 8 attention heads.
#
# infer=True is not optional: load_pretrained_weights is a no-op without it, and the model
# would run on its random initialisation rather than fail.
config = CELLFMConfig(
    img_resize=512,
    img_crop_size=1024,
    cell_image="nucl,er,mt",
    test_cell_image="nucl,er,mt",
    seq_zero_mask_ratio=0.0,
    path_type="Linear",
    prediction="velocity",
    # VAE
    num_down_blocks=3,
    latent_channels=4,
    vae_block_out_channels="128,256,512",
    # CELL-FM
    img_mask_ratio=0,
    cond_out_channels="32,64",
    sample_size=128,
    esm_embedding="esmc_600m",
    encoder_hidden_size=1152,
    max_protein_sequence_len=2048,
    encoder_num_hidden_layers=8,
    num_heads=8,
    dim_head=64,
    dropout=0,
    final_dropout=0,
    encoder_patch_size=8,
    # image generator
    img_generator_num_layers=8,
    img_generator_patch_size=4,
    attention_head_dim=64,
    num_attention_heads=8,
    # image decoder
    img_decoder_num_hidden_layers=4,
    img_decoder_hidden_size=512,
    img_decoder_num_heads=8,
    img_decoder_dim_head=64,
    # checkpoints — vae_512, not vae
    vae_loadcheck_path=hpa("vae_512.bin"),
    loadcheck_path=hpa("cellfm_img2seq.bin"),
    infer=True,
)

# 2.5 GB of CELL-FM weights, plus 2.3 GB for the ESM-C 600M encoder the model is built
# around — the esm package fetches that when the model is constructed, before the
# checkpoint load, even though the checkpoint carries its own copy of those tensors.
t0 = time.time()
model = CELLFMModel(config=config, loss_fn=UniDiffCriterions)
model.to(DEVICE).eval()
vocab = EsmSequenceTokenizer()

# The transformers pin exists to keep this true, so test it rather than trusting a version
# number. When the special-token plumbing moves under esm, mask_token comes back None and
# the failure surfaces much later, inside generation.
if not isinstance(vocab.mask_token, str):
    raise RuntimeError(
        f"vocab.mask_token is {vocab.mask_token!r}, not a string — this build of "
        f"transformers ({transformers.__version__}) does not serve special tokens the way "
        "esm expects, and generation would fail later. Rerun the install cell."
    )

print(f"model {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params, "
      f"ready on {DEVICE} in {time.time() - t0:.0f}s")

In [ ]:
# The two cells the draws are conditioned on. Unlike the other HPA notebook, the protein
# channel is an input here, not something to generate: image_to_sequence encodes it through
# the VAE and asks what sequence would produce that staining.
#
# Both are baked assets, one real HPA cell each, run through the dataset's own preprocessing
# (notebooks/tools/build_pls_assets.py). Using one fixed cell per signal type is what makes
# draws comparable to each other and to the published sequences.
ANCHORS = {
    "nls": ("PPM1G", "Nucleoplasm"),
    "nes": ("DIAPH1", "Cytosol, Plasma membrane"),
}
PLS_TYPES = ("nls", "nes")


def load_anchor(pls_type):
    a = np.load(hpa(f"pls_anchor_{pls_type}.npz"))
    return (torch.from_numpy(a["cell"]).unsqueeze(0).to(DEVICE),
            torch.from_numpy(a["protein"]).unsqueeze(0).to(DEVICE))


anchors = {t: load_anchor(t) for t in PLS_TYPES}

fig, axes = plt.subplots(2, 4, figsize=(10.5, 5.6))
for row, t in enumerate(PLS_TYPES):
    cell, protein = anchors[t]
    gene, locations = ANCHORS[t]
    channels = [("nucleus", cell[0, 0]), ("ER", cell[0, 1]),
                ("microtubules", cell[0, 2]), ("protein", protein[0, 0])]
    for ax, (name, ch) in zip(axes[row], channels):
        ax.imshow((ch.cpu().numpy() + 1) / 2, cmap="gray", vmin=0, vmax=1)
        ax.set_title(name, fontsize=8,
                     color=(NUCLEAR if t == "nls" else CYTO) if name == "protein" else INK)
        ax.set_axis_off()
    axes[row][0].text(-0.08, 0.5, f"{t.upper()}\n{gene}", transform=axes[row][0].transAxes,
                      ha="right", va="center", fontsize=9, fontweight="bold",
                      color=NUCLEAR if t == "nls" else CYTO)
fig.suptitle("the two anchor cells — nuclear protein above, cytosolic below", fontsize=10, y=1.0)
plt.show()

for t in PLS_TYPES:
    print(f"{t}: {ANCHORS[t][0]:7s} ({ANCHORS[t][1]}) — cell {tuple(anchors[t][0].shape)}, "
          f"protein {tuple(anchors[t][1].shape)}")

## 2 · Choose what to generate

A **scaffold** peptide carries the blanks. The model sees `MPSQGSLGAAPPEVAPDSSETEEG` followed
by `LENGTH` masked positions and fills them in; the filled tail is the candidate signal. The
scaffold is inert on purpose — whatever localization appears has to come from the tail.

The defaults are the settings behind the published figure: tail lengths 10 through 25, twenty
draws at each, for both signal types. Lengths matter because a real signal has a size —
classical monopartite NLS motifs are ~7-12 residues and bipartite ones ~16-20 — so scanning a
range lets the model show which size it prefers to put its lysines in.

Cost is one forward pass per masked residue, so the run is `2 × Σ lengths × draws` passes.
The cell prints an estimate before anything starts.

`TEMPERATURE` scales the residue distribution before sampling: below 1 sharpens it towards the
model's favourite residue at each position, above 1 flattens it. 1.0 is the published setting.

In [ ]:
# @title Sampling settings { display-mode: "form" }
LENGTHS = "10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25"  # @param {type:"string"}
DRAWS_PER_LENGTH = 20  # @param {type:"integer"}
TEMPERATURE = 1.0  # @param {type:"number"}
SEED = 6  # @param {type:"integer"}

# Not in the form. The scaffold and the tail position are what the published run used
# (cell_fm/tasks/cell_fm/pls_generation_hpa.py:124), and changing them would put this run on
# a different footing from the figure it reproduces.
SCAFFOLD = "MPSQGSLGAAPPEVAPDSSETEEG"

lengths = [int(x) for x in str(LENGTHS).replace(",", " ").split()]
if not lengths or min(lengths) < 1:
    raise ValueError(f"LENGTHS must be positive integers, got {LENGTHS!r}")

# One forward pass per masked residue, so the cost of a run is its total number of masked
# positions. 0.055 is measured over a full 11,200-position run on an A40 (10.2 min), not
# extrapolated from a short one — a handful of draws reads slower than it should, because
# model warm-up is a larger share of it. The T4 factor of four is the one part still a guess.
SECONDS_PER_POSITION = 0.055
positions = sum(lengths) * DRAWS_PER_LENGTH * len(PLS_TYPES)
n_draws = len(lengths) * DRAWS_PER_LENGTH * len(PLS_TYPES)

# what the published figure was made from: lengths 10-25, 20 draws each, both types
REFERENCE_POSITIONS = sum(range(10, 26)) * 20 * len(PLS_TYPES)

print(f"lengths {lengths}")
print(f"{DRAWS_PER_LENGTH} draws each, {len(PLS_TYPES)} signal types -> {n_draws} peptides")
print(f"{positions:,} masked positions = {positions:,} forward passes")
print(f"\nestimate: {positions * SECONDS_PER_POSITION / 60:.0f} min on an A40, "
      f"~{positions * SECONDS_PER_POSITION * 4 / 60:.0f} min on a T4.")
if positions >= REFERENCE_POSITIONS:
    print("These are the reference settings, so the figure below reproduces the published one.")
else:
    print(f"Below the reference settings ({REFERENCE_POSITIONS:,} positions); the figure will "
          "be noisier than the published one.")

## 3 · Stage 1 — fill the blanks

Each draw masks the tail afresh and fills it one position at a time, in random order, each
residue conditioned on the anchor image and on everything decided so far. The loop runs the
NLS anchor first, then the NES one; nothing else changes between them.

**Every draw starts from a fully masked tail.** The cell below re-tokenizes the prompt each
time, and does so explicitly rather than relying on the model code to leave its input alone.

That is not hypothetical. The first published sequences were made without it: `oaardm_sample`
filled the token tensor in place and handed the same object back, so a loop that reused it
resampled the *previous* peptide instead of starting over, and the draws came out as a chain —
consecutive sequences differed by one or two residues out of ten. Measured on an A40 at ten
residues, independent draws differ by about six and were six-unique-in-six, against
four-in-six for the chain, which produced two exact duplicate pairs.

Six rather than ten, because the model is confident about what a nuclear signal is made of:
even independent draws keep reaching for the same lysines and arginines. That shared
composition is the result. The near-identity of a chain is not.

It also makes the frequency test below legitimate: a chi-squared over correlated draws counts
the same evidence repeatedly and reports a smaller p-value than the data earns.

In [ ]:
import random

from tqdm.auto import tqdm


def seed_everything(seed):
    """Matches cell_fm/pipeline/accelerator/trainer.py, which the offline task calls.

    numpy matters here beyond reproducibility of the weights: order="random" shuffles the
    fill order with np.random.shuffle, so the sampling path itself depends on it.
    """
    torch.cuda.manual_seed_all(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


@torch.no_grad()
def generate_signal(num_aa, seed, cell_img, protein_img, temperature=1.0):
    """One independent candidate signal of num_aa residues.

    Re-tokenizing here is the whole point: it is what makes this draw independent of the
    last one. See the note above.
    """
    seed_everything(seed)
    tokens = encoding.tokenize_sequence(
        SCAFFOLD + "<mask>" * num_aa, vocab, True).unsqueeze(0).to(DEVICE)
    mask = tokens == vocab.mask_token_id

    out = model.image_to_sequence(
        tokens, mask, protein_img, cell_img,
        order="random", temperature=temperature, progress=False,
    )
    # decode_sequence strips <cls>/<eos>, so the last num_aa characters are exactly the tail
    full = decoding.decode_sequence(out.squeeze(), vocab)
    return full, full[-num_aa:]


if str(DEVICE) == "cpu":
    print("No GPU — this will take hours. Runtime > Change runtime type > T4 GPU.\n")

records = []
t0 = time.time()
with tqdm(total=n_draws, desc="draws", unit="pep") as bar:
    for type_index, pls_type in enumerate(PLS_TYPES):
        cell_img, protein_img = anchors[pls_type]
        for num_aa in lengths:
            for k in range(DRAWS_PER_LENGTH):
                # the type offset keeps NLS and NES off identical mask orders, so the two
                # panels are independent samples rather than the same draw twice
                seed = SEED + 100_000 * type_index + 1_000 * num_aa + k
                full, signal = generate_signal(
                    num_aa, seed, cell_img, protein_img, TEMPERATURE)
                records.append({"pls_type": pls_type, "num_aa": num_aa, "draw": k + 1,
                                "signal": signal, "full_sequence": full})
                bar.update(1)
        bar.set_postfix_str(f"{pls_type} done")
gen_seconds = time.time() - t0

generated = pd.DataFrame.from_records(records)
print(f"{len(generated)} peptides in {gen_seconds / 60:.1f} min — "
      f"{gen_seconds / max(1, positions):.3f}s per position")

## 4 · The candidate signals

What came out. Duplicates are worth watching: at a short length the model has few ways to say
"nucleus", so repeats mean it is confident rather than that something has gone wrong.

The K+R column is the first look at whether the two anchors did anything. Lysine and arginine
are what a nuclear localization signal is built from, and the human proteome sits near 11.4%
of residues — so NLS should land far above that line and NES well below it, from nothing but
the choice of conditioning cell.

In [ ]:
pd.set_option("display.max_colwidth", 40)


def kr_content(seqs):
    """Fraction of residues that are lysine or arginine, averaged over draws."""
    return float(np.mean([sum(c in "KR" for c in s) / len(s) for s in seqs]))


rows = []
for pls_type in PLS_TYPES:
    g = generated[generated.pls_type == pls_type]
    rows.append({"pls_type": pls_type, "draws": len(g), "unique": g.signal.nunique(),
                 "lengths": g.num_aa.nunique(), "K+R": f"{kr_content(g.signal):.1%}"})
print(pd.DataFrame(rows).to_string(index=False))
print("\nhuman proteome K+R: 11.4%")

display(generated.groupby("pls_type").head(4)[["pls_type", "num_aa", "draw", "signal"]])

## 5 · Stage 2 — what the model reached for

A candidate signal is judged by composition. This is the analysis from
`cell_fm/tasks_local/analysis/ana_pls_gen_dis_with_baseline.py`: count residues across all
draws of a type, compare against the human proteome, and test each residue with a chi-squared.

The proteome baseline is 12,894 HPA proteins and 7.9 M residues, shipped as twenty counts
rather than the 60 MB table they came from — the test needs counts, so counts are all that
has to travel.

The two panels are the point. NLS on its own could be an artefact of the scaffold, of the
anchor, or of the sampler; NES is the control that rules that out. Both panels come from the
same model, the same scaffold and the same code, differing only in which cell was shown — so
if they come back with opposite chemistry, that difference is the model reading the image.

One deviation from the published figure, deliberate: its two panels have independent x-scales,
so a longer bar on one side can stand for a smaller shift than a shorter bar on the other. Here
they share a scale, and bar lengths mean the same thing in both panels. The exported copy keeps
the published settings.

In [ ]:
import json
from collections import Counter

from matplotlib.ticker import MaxNLocator
from scipy.stats import chi2_contingency

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
# grouped by chemistry: basic, acidic, polar, small, hydrophobic, aromatic, special
AA_ORDER = ["R", "H", "K", "D", "E", "S", "T", "N", "Q", "G",
            "A", "V", "I", "L", "M", "C", "F", "Y", "W", "P"]
PANEL_COLOURS = {"nls": "#2196F3", "nes": "#F44336"}


def aa_counts(seqs):
    counter = Counter()
    for s in seqs:
        counter.update(c for c in s if c in AMINO_ACIDS)
    return {aa: counter.get(aa, 0) for aa in AMINO_ACIDS}


def shifts_and_pvals(sig_counts, base_counts):
    """Percentage-point shift per residue, with a per-residue chi-squared against baseline.

    The chi-squared is over counts, not frequencies — the same twenty 2x2 tables the offline
    script builds. Feeding it frequencies would make every p-value a function of how the
    numbers were scaled rather than of how much data there is.
    """
    sig_tot, base_tot = sum(sig_counts.values()), sum(base_counts.values())
    shifts, pvals = [], []
    for aa in AA_ORDER:
        shifts.append((sig_counts[aa] / sig_tot - base_counts[aa] / base_tot) * 100)
        _, p, _, _ = chi2_contingency(np.array([
            [sig_counts[aa], sig_tot - sig_counts[aa]],
            [base_counts[aa], base_tot - base_counts[aa]],
        ]))
        pvals.append(p)
    return shifts, pvals


def plot_combined(counts, base_counts, sharex=True, paper=False):
    """The published two-panel figure (ana_pls_gen_dis_with_baseline.py:89).

    paper=True takes its settings — 10x15 at font 25, independent x-scales — which are sized
    for a printed panel and unreadable inline. The default is the notebook's own palette with
    the panels on a shared x-scale. The one thing paper mode does not copy is the tick
    density; see below.
    """
    style = {"font.family": "sans-serif", "font.size": 25, "axes.titlesize": 30,
             "axes.labelsize": 25, "xtick.labelsize": 25, "ytick.labelsize": 25}
    y = np.arange(len(AA_ORDER))

    with plt.rc_context(style if paper else {}):
        fig, axes = plt.subplots(1, 2, figsize=(10, 15) if paper else (8.5, 9),
                                 dpi=150, sharey=True, sharex=sharex)
        for ax, pls_type in zip(axes, PLS_TYPES):
            colour = PANEL_COLOURS[pls_type]
            shifts, _ = shifts_and_pvals(counts[pls_type], base_counts)
            ax.barh(y, shifts, height=0.65, zorder=2,
                    color=[colour if v > 0 else "#aaaaaa" for v in shifts])
            ax.axvline(0, color="#888888", linewidth=0.8)
            ax.set_xlabel(pls_type.upper(), color=colour)
            ax.spines[["top", "right"]].set_visible(False)
            ax.xaxis.grid(True, linestyle="--", linewidth=0.5, color="#eeeeee", zorder=0)
            ax.set_axisbelow(True)
            ax.tick_params(axis="y", length=0)
            if paper:
                # at font 25 in a half-width panel, matplotlib's default tick count runs the
                # labels into each other and the axis reads as one smear of digits. The
                # published figure escaped it only because its data landed on a wide round
                # range; a shorter run does not.
                ax.xaxis.set_major_locator(
                    MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))

        axes[0].set_yticks(y)
        axes[0].set_yticklabels(AA_ORDER)
        axes[0].set_ylabel("Amino acid")
        axes[0].invert_yaxis()  # keep AA_ORDER top-to-bottom
        fig.suptitle("Δ frequency (pp) vs. proteome baseline")
        fig.tight_layout()
    return fig


baseline = json.load(open(hpa("proteome_aa_counts.json")))
base_counts = baseline["counts"]
counts = {t: aa_counts(generated[generated.pls_type == t].signal.tolist())
          for t in PLS_TYPES}

fig = plot_combined(counts, base_counts)
plt.show()

for pls_type in PLS_TYPES:
    shifts, pvals = shifts_and_pvals(counts[pls_type], base_counts)
    top = sorted(zip(AA_ORDER, shifts, pvals), key=lambda t: -t[1])[:4]
    n = int(sum(counts[pls_type].values()))
    print(f"{pls_type.upper()} — {n:,} residues, most enriched:")
    for aa, s, p in top:
        flag = "" if p < 0.05 / len(AA_ORDER) else "   (not significant after Bonferroni)"
        print(f"   {aa}  {s:+6.2f} pp   p = {p:.1e}{flag}")

## 6 · Export

| File | What it holds |
| --- | --- |
| `pls_generated.csv` | one row per draw: signal type, tail length, draw number, the signal, and the full peptide it came from |
| `nls_nes_vs_baseline.png` | the figure above, re-rendered at the published figure's size, font and independent x-scales, with the x ticks thinned so they stay legible at that font size |

In [ ]:
def export(generated, counts, base_counts, stem="pls"):
    """Write the table and the figure, then offer both for download.

    The figure is re-rendered rather than reused: the inline one is sized to be readable in
    a notebook, and this one matches the published figure (10x15 at font 25, dpi 300,
    transparent, independent x-scales) so it can go straight into a panel.
    """
    csv_path, png_path = f"{stem}_generated.csv", "nls_nes_vs_baseline.png"
    generated.to_csv(csv_path, index=False)

    paper_fig = plot_combined(counts, base_counts, sharex=False, paper=True)
    paper_fig.savefig(png_path, dpi=300, bbox_inches="tight", transparent=True)
    plt.close(paper_fig)

    download_buttons(csv_path, png_path)
    return csv_path, png_path


def download_buttons(*paths):
    """One button per file, rather than firing the downloads the moment the cell runs.

    The Output widget is load-bearing: files.download runs JavaScript in a live output
    context, and a click handler has none of its own, so calling it straight from on_click
    silently does nothing.
    """
    for path in paths:
        print(f"{path}  {os.path.getsize(path) / 1e6:.2f} MB")

    try:
        import ipywidgets as widgets
        from google.colab import files
        from IPython.display import display
    except ImportError:
        print("\nno download button outside Colab — both files are in the working directory")
        return

    sink = widgets.Output()

    def fetch(path):
        with sink:
            files.download(path)

    buttons = [
        widgets.Button(description=f"Download {os.path.basename(p)}",
                       icon="download", layout=widgets.Layout(width="auto"))
        for p in paths
    ]
    for button, path in zip(buttons, paths):
        button.on_click(lambda _, p=path: fetch(p))

    display(widgets.HBox(buttons), sink)


_ = export(generated, counts, base_counts)